In [ ]:
!pip install timm opencv-python torch torchvision

# PolyGotFake Training

In [ ]:
#!/usr/bin/env python3
"""

Highlights:
- Video branch: EfficientNet-B0 feature extractor + LSTM temporal module (as requested)
- Audio branch: Log-Mel spectrogram extraction (librosa + ffmpeg fallback) + TCN
- Fusion: Game-theoretic fusion (Shapley-Replicator Fusion) + fallback MLP
- Deep supervision (frame + seq losses) reused for video branch
- FocalLoss supported for imbalanced classes
- Dataset pairs video files with audio (extract audio from same .mp4)

"""

import os
import sys
import copy
import math
import random
import tempfile
import subprocess
from typing import List, Tuple, Optional

import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms
import torchvision.models as models
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix
from tqdm import tqdm

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------
# Minimal configurable settings (you can edit or parse CLI args)
# ---------------------------
CONFIG = {
    'data_root': '/kaggle/input/polyglotfake-rs/PolyGlotFake',
    'model_save_path': './checkpoints_multimodal',
    'num_frames': 8,
    'frame_size': 224,
    'batch_size': 8,
    'num_workers': 2,
    'dropout_rate': 0.5,
    'use_spatial_dropout': False,
    'use_temporal_dropout': True,
    'tcn_hidden_dim': 256,
    'tcn_channels': [256],
    'tcn_kernel_size': 3,
    'use_attention': True,
    'layers_to_freeze': 2,
    'use_focal_loss': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    'learning_rate': 2e-4,
    'weight_decay': 1e-4,
    'scheduler_factor': 0.5,
    'scheduler_patience': 3,
    'num_epochs': 20,
    'early_stopping_patience': 6,
    'aux_loss_weights': {'seq': 0.5, 'frame': 0.5},
    'random_seed': 42,
    'audio_num_frames': 128,
    'audio_n_mels': 64,
    # optional LSTM config defaults (safe fallback)
    'lstm_hidden_dim': 128,
    'use_transformer': False,
    'transformer_nhead': 8,
    'transformer_nlayers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# reproducibility
torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])
random.seed(CONFIG['random_seed'])

# ---------------------------
# Utilities: plotting, loss
# ---------------------------

def plot_confusion_matrix(cm, epoch=None, save_path=CONFIG['model_save_path']):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Fake', 'Predicted Real'],
                yticklabels=['True Fake', 'True Real'])
    plt.xlabel('Predicted Labels'); plt.ylabel('True Labels')
    title = 'Confusion Matrix'
    if epoch is not None:
        title += f' - Epoch {epoch+1}'
    plt.title(title)
    if epoch is not None:
        os.makedirs(save_path, exist_ok=True)
        plt.savefig(os.path.join(save_path, f'confusion_matrix_epoch_{epoch+1}.png'))
    plt.close()


def plot_metrics(metrics, save_path=CONFIG['model_save_path']):
    os.makedirs(save_path, exist_ok=True)
    epochs = range(1, len(metrics['train_loss']) + 1)
    plt.figure(figsize=(12,8))
    plt.subplot(2,2,1)
    plt.plot(epochs, metrics['train_loss'], '-', label='Training Loss')
    plt.plot(epochs, metrics['val_loss'], '-', label='Validation Loss')
    plt.title('Loss'); plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
    plt.subplot(2,2,2)
    plt.plot(epochs, metrics['train_acc'], '-', label='Training Acc')
    plt.plot(epochs, metrics['val_acc'], '-', label='Validation Acc')
    plt.title('Accuracy'); plt.xlabel('Epochs'); plt.legend(); plt.grid(True)
    plt.subplot(2,2,3)
    plt.plot(epochs, metrics['val_auc'], '-')
    plt.title('Val AUC'); plt.xlabel('Epochs'); plt.grid(True)
    plt.subplot(2,2,4)
    plt.plot(epochs, metrics['val_precision'], '-', label='Precision (Val)')
    plt.plot(epochs, metrics['val_recall'], '-', label='Recall (Val)')
    plt.title('Precision & Recall'); plt.xlabel('Epochs'); plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'training_metrics.png'))
    plt.close()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Make sure both inputs and targets are 1D tensors with the same shape.
        # This handles cases where logits are returned as (B,) but labels come as (B,1).
        inputs = inputs.view(-1)
        targets = targets.view(-1).type_as(inputs)

        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * (1 - pt)**self.gamma * BCE_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ---------------------------
# Frame extraction (robust)
# ---------------------------
import cv2

def extract_frames(video_path, num_frames, frame_size):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return np.zeros((num_frames, frame_size, frame_size, 3), dtype=np.uint8)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return np.zeros((num_frames, frame_size, frame_size, 3), dtype=np.uint8)

    indices = np.linspace(0, max(total_frames-1, 0), num_frames, endpoint=True, dtype=int)
    frames = []
    for idx in indices:
        if idx >= total_frames:
            idx = total_frames - 1
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (frame_size, frame_size))
            frames.append(frame)
        else:
            # append zeros
            frames.append(np.zeros((frame_size, frame_size, 3), dtype=np.uint8))
    cap.release()
    while len(frames) < num_frames:
        frames.append(frames[-1].copy() if frames else np.zeros((frame_size, frame_size, 3), dtype=np.uint8))
    return np.array(frames[:num_frames])

# ---------------------------
# Audio processor (librosa + ffmpeg fallback)
# ---------------------------
class LogMelSpectrogramProcessor:
    def __init__(self, sample_rate=16000, num_frames=128, n_mels=64, n_fft=1024, hop_length=256, window='hamming'):
        self.sample_rate = sample_rate
        self.num_frames = num_frames
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.window = window
        self.video_exts = {'.mp4', '.mkv', '.mov', '.avi', '.webm'}

    def extract_log_mel_spectrogram(self, audio):
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        mel_spec = librosa.feature.melspectrogram(
            y=audio,
            sr=self.sample_rate,
            n_mels=self.n_mels,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            window=self.window,
            power=2.0
        )
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        return self._frame_spectrogram(log_mel_spec)

    def _frame_spectrogram(self, spectrogram):
        time_steps = spectrogram.shape[1]
        if time_steps >= self.num_frames:
            framed_spec = spectrogram[:, :self.num_frames]
        else:
            pad_width = self.num_frames - time_steps
            framed_spec = np.pad(spectrogram, ((0, 0), (0, pad_width)), mode='constant')
        # return (num_frames, n_mels)
        return framed_spec.T

    def _extract_audio_with_ffmpeg(self, file_path):
        tmp_wav = None
        try:
            tmpf = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
            tmp_wav = tmpf.name
            tmpf.close()
            cmd = [
                'ffmpeg', '-y', '-nostdin', '-i', file_path,
                '-ar', str(self.sample_rate), '-ac', '1', '-vn', tmp_wav
            ]
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            return tmp_wav
        except Exception as e:
            if tmp_wav and os.path.exists(tmp_wav):
                try:
                    os.remove(tmp_wav)
                except:
                    pass
            return None

    def process_audio_file(self, file_path):
        try:
            ext = os.path.splitext(file_path)[1].lower()
            if ext in self.video_exts:
                tmp_wav = self._extract_audio_with_ffmpeg(file_path)
                if tmp_wav is None:
                    return None
                try:
                    waveform, _ = librosa.load(tmp_wav, sr=self.sample_rate, mono=True)
                finally:
                    try:
                        os.remove(tmp_wav)
                    except:
                        pass
            else:
                waveform, _ = librosa.load(file_path, sr=self.sample_rate, mono=True)
            spec = self.extract_log_mel_spectrogram(waveform)
            return spec.astype(np.float32)
        except Exception as e:
            print(f"Error processing audio {file_path}: {e}")
            return None

# ---------------------------
# Temporal modules (TCN) - reuse earlier PyTorch implementation
# ---------------------------
class ChannelNormalization(nn.Module):
    def __init__(self):
        super(ChannelNormalization, self).__init__()
    def forward(self, x):
        # x: (batch, channels, seq)
        max_values = torch.max(torch.abs(x), dim=2, keepdim=True)[0] + 1e-5
        return x / max_values

class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
    def forward(self, x):
        x = self.conv(x)
        if self.padding > 0:
            x = x[:, :, :-self.padding]
        return x


def wavenet_activation(x):
    tanh_out = torch.tanh(x)
    sigm_out = torch.sigmoid(x)
    return tanh_out * sigm_out

class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout_rate=0.1):
        super(TCNBlock, self).__init__()
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel_size, dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.bn3 = nn.BatchNorm1d(out_channels)
        self.channel_norm = ChannelNormalization()
        self.dropout = nn.Dropout(dropout_rate)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = wavenet_activation(out)
        out = self.channel_norm(out)
        out = self.bn3(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(residual)
        out = out + residual
        out = wavenet_activation(out)
        return out

class TCN(nn.Module):
    def __init__(self, input_size, num_channels, kernel_size=3, dropout_rate=0.1):
        super(TCN, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(TCNBlock(in_channels, out_channels, kernel_size, dilation_size, dropout_rate))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        # x: (batch, seq_len, features) -> transpose to (batch, features, seq_len)
        x = x.transpose(1, 2)
        x = self.network(x)
        x = x.transpose(1, 2)
        return x

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
    def forward(self, seq):
        attn_logits = self.attention(seq)
        attn_weights = F.softmax(attn_logits, dim=1)
        context = torch.sum(attn_weights * seq, dim=1)
        return context, attn_weights

# ---------------------------
# Video branch: REPLACED to match supplied DeepfakeDetectorFusion (EfficientNet-B0 + LSTM + Attention)
# ---------------------------
class VideoBranch(nn.Module):
    def __init__(self, config):
        super(VideoBranch, self).__init__()
        self.config = config

        # RGB backbone: EfficientNet-B0
        # Use torchvision's efficientnet_b0; this gives feature dim = 1280
        efficientnet = models.efficientnet_b0(weights='DEFAULT')
        feature_blocks = efficientnet.features

        # freeze early feature blocks per config
        layers_to_freeze = config.get('layers_to_freeze', 3)
        ct = 0
        for idx, child in enumerate(feature_blocks.children()):
            if ct < layers_to_freeze:
                for param in child.parameters():
                    param.requires_grad = False
            else:
                for param in child.parameters():
                    param.requires_grad = True
            ct += 1

        # combine features + avgpool to get (B*T, feat_dim, 1, 1)
        self.rgb_feature_extractor = nn.Sequential(efficientnet.features, efficientnet.avgpool)
        self.feature_dim = 1280

        self.spatial_dropout = nn.Dropout(p=config['dropout_rate']) if config['use_spatial_dropout'] else nn.Identity()

        # temporal module: transformer option or LSTM
        self.use_transformer = config.get('use_transformer', False)
        if self.use_transformer:
            n_heads = max(1, config.get('transformer_nhead', 8))
            encoder_layer = nn.TransformerEncoderLayer(d_model=self.feature_dim, nhead=n_heads, batch_first=True,
                                                       dim_feedforward=self.feature_dim*2, dropout=config['dropout_rate'])
            self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.get('transformer_nlayers', 2))
            lstm_output_dim = self.feature_dim
        else:
            lstm_hidden = config.get('lstm_hidden_dim', config.get('tcn_hidden_dim', 128))
            self.lstm = nn.LSTM(
                input_size=self.feature_dim,
                hidden_size=lstm_hidden,
                num_layers=2,
                batch_first=True,
                bidirectional=True,
                dropout=config['dropout_rate'] if config['use_temporal_dropout'] and 2 > 1 else 0
            )
            lstm_output_dim = lstm_hidden * 2

        # temporal attention
        self.use_attention = config.get('use_attention', True)
        if self.use_attention:
            self.temporal_attention = TemporalAttention(lstm_output_dim)

        # heads
        self.classifier_main_fc = nn.Linear(lstm_output_dim, config.get('lstm_hidden_dim', config.get('tcn_hidden_dim', 128)))
        self.classifier_main_dropout = nn.Dropout(config['dropout_rate'])
        self.classifier_main_out = nn.Linear(config.get('lstm_hidden_dim', config.get('tcn_hidden_dim', 128)), 1)

        self.frame_proj = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim // 2),
            nn.ReLU(),
            nn.Linear(self.feature_dim // 2, 1)
        )

        self.classifier_seq = nn.Sequential(
            nn.Linear(lstm_output_dim, config.get('lstm_hidden_dim', config.get('tcn_hidden_dim', 128))),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(config.get('lstm_hidden_dim', config.get('tcn_hidden_dim', 128)), 1)
        )

    def forward(self, rgb_x):
        """
        rgb_x: (B, T, C, H, W)
        returns: main_logits (B,), seq_logits (B,), frame_logits (B,T), penultimate_features (B, hidden_dim)
        """
        B, T, C, H, W = rgb_x.size()
        rgb_flat = rgb_x.view(B * T, C, H, W)
        rgb_feats = self.rgb_feature_extractor(rgb_flat)  # (B*T, feat_dim, 1, 1)
        rgb_feats = rgb_feats.view(B, T, -1)             # (B,T,feat_dim)

        if self.config['use_spatial_dropout']:
            rgb_feats = self.spatial_dropout(rgb_feats)

        # frame-level logits
        frame_logits = self.frame_proj(rgb_feats).squeeze(-1)  # (B,T)

        # temporal modeling
        if self.use_transformer:
            temporal_out = self.temporal_encoder(rgb_feats)  # (B,T,feat_dim)
            seq_pooled = temporal_out.mean(dim=1)
            seq_logits = self.classifier_seq(seq_pooled).squeeze(-1)
            if self.use_attention:
                context, attn_weights = self.temporal_attention(temporal_out)
            else:
                context = temporal_out[:, -1, :]
        else:
            lstm_out, _ = self.lstm(rgb_feats)  # (B,T,lstm_output_dim)
            seq_pooled = lstm_out.mean(dim=1)
            seq_logits = self.classifier_seq(seq_pooled).squeeze(-1)
            if self.use_attention:
                context, attn_weights = self.temporal_attention(lstm_out)
            else:
                context = lstm_out[:, -1, :]

        # main classifier + penultimate (after ReLU)
        penultimate = F.relu(self.classifier_main_fc(context))  # (B, hidden_dim)
        dropped = self.classifier_main_dropout(penultimate)
        main_logits = self.classifier_main_out(dropped).squeeze(-1)

        return main_logits, seq_logits, frame_logits, penultimate

# ---------------------------
# Audio branch (PyTorch TCN)
# ---------------------------
class AudioBranch(nn.Module):
    def __init__(self, config, input_n_mels=64):
        super(AudioBranch, self).__init__()
        self.config = config
        self.input_n_mels = input_n_mels
        # project mel features to higher-dim before TCN
        self.pre_linear = nn.Linear(input_n_mels, config['tcn_hidden_dim'])
        tcn_channels = config.get('tcn_channels', [256])
        self.tcn = TCN(input_size=config['tcn_hidden_dim'], num_channels=tcn_channels,
                       kernel_size=config.get('tcn_kernel_size', 3), dropout_rate=config['dropout_rate'])
        self.temporal_attention = TemporalAttention(tcn_channels[-1]) if config.get('use_attention', True) else None
        self.seq_classifier = nn.Sequential(
            nn.Linear(tcn_channels[-1], config['tcn_hidden_dim']),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(config['tcn_hidden_dim'], 1)
        )
        # penultimate projection
        self.penultimate_proj = nn.Linear(tcn_channels[-1], config['tcn_hidden_dim'])

    def forward(self, mel_specs):
        # mel_specs: (B, seq_len, n_mels)
        x = self.pre_linear(mel_specs)  # (B, seq_len, hidden)
        tcn_out = self.tcn(x)  # (B, seq_len, tcn_out)
        seq_pooled = tcn_out.mean(dim=1)
        seq_logits = self.seq_classifier(seq_pooled).squeeze(-1)
        if self.temporal_attention is not None:
            context, _aw = self.temporal_attention(tcn_out)
        else:
            context = tcn_out[:, -1, :]
        penultimate = F.relu(self.penultimate_proj(context))
        return seq_logits, penultimate

# ---------------------------
# Game-theoretic fusion and MultiModalDetector (ADAPTED for heterogeneous penultimate dims)
# ---------------------------
class GameTheoreticFusion(nn.Module):
    """
    Game-theoretic fusion that supports different penultimate dims for video and audio branches.
    Heads sizes are adapted to video_dim and audio_dim.
    """
    def __init__(self, video_dim: int, audio_dim: int, temperature=1.0, replicator_steps=5, mcr_weight=0.1):
        super(GameTheoreticFusion, self).__init__()
        self.temperature = temperature
        self.replicator_steps = replicator_steps
        self.mcr_weight = mcr_weight
        self.video_dim = video_dim
        self.audio_dim = audio_dim

        # choose small hidden sizes for heads
        v_hidden = max(8, video_dim // 2)
        a_hidden = max(8, audio_dim // 2)
        both_hidden = max(16, (video_dim + audio_dim) // 2)

        # heads to produce coalition logits
        self.head_v = nn.Sequential(
            nn.Linear(video_dim, v_hidden),
            nn.ReLU(),
            nn.Linear(v_hidden, 1)
        )
        self.head_a = nn.Sequential(
            nn.Linear(audio_dim, a_hidden),
            nn.ReLU(),
            nn.Linear(a_hidden, 1)
        )
        # joint head takes concatenated pen features
        self.head_both = nn.Sequential(
            nn.Linear(video_dim + audio_dim, both_hidden),
            nn.ReLU(),
            nn.Linear(both_hidden, 1)
        )

    def forward(self, pen_v, pen_a, logits=None):
        """
        pen_v: (B, video_dim)
        pen_a: (B, audio_dim)
        returns: fused_logit (B,), aux dict {'mcr':loss, 'weights':W (B,3)}
        """
        # coalition logits (B,)
        v_only = self.head_v(pen_v).view(-1)
        a_only = self.head_a(pen_a).view(-1)
        both_in = torch.cat([pen_v, pen_a], dim=1)
        both = self.head_both(both_in).view(-1)

        # Shapley-style approximations (for 2 players, closed-form)
        phi_v = 0.5 * (v_only + both - a_only)
        phi_a = 0.5 * (a_only + both - v_only)
        phi_both = torch.relu(both - 0.5 * (v_only + a_only))

        # non-negative utilities
        u_v = torch.relu(phi_v)
        u_a = torch.relu(phi_a)
        u_b = torch.relu(phi_both)

        # replicator / multiplicative weights (per-sample)
        batch_size = u_v.size(0)
        device = u_v.device
        W = torch.ones((batch_size, 3), device=device) / 3.0
        U = torch.stack([u_v, u_a, u_b], dim=1)  # (B,3)
        eta = 0.5 / (self.temperature + 1e-8)
        for _ in range(self.replicator_steps):
            W = W * torch.exp(eta * U)
            W = W / (W.sum(dim=1, keepdim=True) + 1e-8)

        w_v = W[:, 0]
        w_a = W[:, 1]
        w_b = W[:, 2]

        # fuse
        fused_logit = w_v * v_only + w_a * a_only + w_b * both

        # multimodal competition regularizer (encourage balanced cooperation)
        mcr_loss = ( (w_v - w_a).abs().mean() )
        aux = {'mcr': mcr_loss * self.mcr_weight, 'weights': W}
        return fused_logit, aux

class MultiModalDetector(nn.Module):
    def __init__(self, config, audio_n_mels=64):
        super(MultiModalDetector, self).__init__()
        self.video_branch = VideoBranch(config)
        self.audio_branch = AudioBranch(config, input_n_mels=audio_n_mels)

        # infer penultimate dims from branches
        # video penultimate dim is classifier_main_fc.out_features
        video_pen_dim = self.video_branch.classifier_main_fc.out_features
        # audio penultimate dim is penultimate_proj.out_features
        audio_pen_dim = self.audio_branch.penultimate_proj.out_features

        # game-theoretic fusion that supports different dims
        self.gt_fusion = GameTheoreticFusion(video_dim=video_pen_dim, audio_dim=audio_pen_dim,
                                             temperature=1.0, replicator_steps=6, mcr_weight=0.5)

        # fallback fusion MLP (match input dim = video_pen_dim + audio_pen_dim)
        fusion_in = video_pen_dim + audio_pen_dim
        fusion_hidden = max(config.get('tcn_hidden_dim', 256), fusion_in // 2)
        self.fallback_fusion = nn.Sequential(
            nn.Linear(fusion_in, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(fusion_hidden, 1)
        )

    def forward(self, rgb_x, mel_specs):
        main_v, seq_v, frame_logits, pen_v = self.video_branch(rgb_x)
        seq_a, pen_a = self.audio_branch(mel_specs)
        fusion_logit, aux = self.gt_fusion(pen_v, pen_a, logits=None)
        fallback_in = torch.cat([pen_v, pen_a], dim=1)
        fallback_logit = self.fallback_fusion(fallback_in).squeeze(-1)
        return {
            'video_logit': main_v,
            'video_seq_logit': seq_v,
            'frame_logits': frame_logits,
            'audio_seq_logit': seq_a,
            'fusion_logit': fusion_logit,
            'fusion_aux': aux,
            'fallback_logit': fallback_logit
        }

# ---------------------------
# Dataset: pair video with audio (search for audio with same basename or extract audio from video)
# ---------------------------
class MultiModalDataset(Dataset):
    def __init__(self, video_paths: List[str], labels: List[int], config, audio_search_dirs: Optional[List[str]]=None, transform=None):
        self.video_paths = video_paths
        self.labels = np.array(labels, dtype=np.int64)
        self.config = config
        self.transform = transform
        self.processor = LogMelSpectrogramProcessor(sample_rate=16000, num_frames=config['audio_num_frames'], n_mels=config['audio_n_mels'])
        self.audio_search_dirs = audio_search_dirs or []

    def __len__(self):
        return len(self.video_paths)

    def find_audio_for_video(self, video_path):
        return video_path

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = int(self.labels[idx])
        frames = extract_frames(video_path, self.config['num_frames'], self.config['frame_size'])
        if self.transform:
            try:
                frames_t = [self.transform(f) for f in frames]
                rgb_tensor = torch.stack(frames_t)  # (T, C, H, W)
            except Exception as e:
                print(f"Transform error on {video_path}: {e}")
                rgb_tensor = torch.from_numpy(frames).permute(0,3,1,2).float() / 255.0
        else:
            rgb_tensor = torch.from_numpy(frames).permute(0,3,1,2).float() / 255.0
        audio_path = self.find_audio_for_video(video_path)
        mel = self.processor.process_audio_file(audio_path)
        if mel is None:
            mel = np.zeros((self.config['audio_num_frames'], self.config['audio_n_mels']), dtype=np.float32)
        mel_tensor = torch.from_numpy(mel).float()  # (seq_len, n_mels)
        return rgb_tensor, mel_tensor, torch.tensor(label, dtype=torch.float), video_path

# ---------------------------
# Dataset preparation helper (collect video files same as your original prepare_dataset)
# ---------------------------
def collect_videos(data_root):
    real_dirs = ['real/ar','real/en','real/es','real/fr','real/ja','real/ru','real/zh']
    fake_dirs = ['fake/to_ar','fake/to_en','fake/to_es','fake/to_fr','fake/to_ja','fake/to_ru','fake/to_zh']
    all_video_files = []
    for dir_name in real_dirs:
        dir_path = os.path.join(data_root, dir_name)
        if not os.path.isdir(dir_path):
            continue
        for file_name in os.listdir(dir_path):
            if file_name.lower().endswith('.mp4'):
                all_video_files.append((os.path.join(dir_path, file_name), 1))
    for dir_name in fake_dirs:
        dir_path = os.path.join(data_root, dir_name)
        if not os.path.isdir(dir_path):
            continue
        for file_name in os.listdir(dir_path):
            if file_name.lower().endswith('.mp4'):
                all_video_files.append((os.path.join(dir_path, file_name), 0))
    if not all_video_files:
        raise ValueError(f"No video files found under {data_root}")
    video_paths = [p for p,_ in all_video_files]
    labels = [l for _,l in all_video_files]
    return video_paths, labels

# ---------------------------
# Training & evaluation
# ---------------------------

def get_class_weights(labels):
    class_counts = np.bincount(labels)
    if len(class_counts) < 2 or class_counts[0]==0 or class_counts[1]==0:
        return torch.FloatTensor([1.0, 1.0])
    total = len(labels)
    return torch.FloatTensor(total / (len(class_counts) * class_counts))


def train_multimodal(model, train_loader, val_loader, config):
    device = config['device']
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=config['scheduler_factor'], patience=config['scheduler_patience'])

    criterion = FocalLoss(alpha=config['focal_alpha'], gamma=config['focal_gamma']) if config['use_focal_loss'] else nn.BCEWithLogitsLoss()

    best_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    scaler = GradScaler()

    metrics = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_precision': [], 'val_recall': [], 'val_auc': []}

    for epoch in range(config['num_epochs']):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} Training", leave=False)
        for batch in pbar:
            rgb, mel, labels, _paths = batch
            rgb = rgb.to(device)
            mel = mel.to(device)
            labels = labels.to(device).view(-1).float()
            optimizer.zero_grad()
            with autocast():
                outputs = model(rgb, mel)
                # outputs: dict with video_logit, video_seq_logit, frame_logits, audio_seq_logit, fusion_logit, fusion_aux
                # compute losses
                loss_main = criterion(outputs['fusion_logit'].view(-1), labels)
                loss_video_main = criterion(outputs['video_logit'].view(-1), labels)
                loss_seq = criterion(outputs['video_seq_logit'].view(-1), labels)
                # frame-level: flatten
                frame_logits = outputs['frame_logits']
                labels_frame = labels.unsqueeze(1).repeat(1, frame_logits.size(1)).to(frame_logits.device)
                loss_frame = criterion(frame_logits.view(-1), labels_frame.view(-1))

                # fusion auxiliary (MCR) regularizer if present
                aux = outputs.get('fusion_aux', None)
                if aux is not None and isinstance(aux, dict) and 'mcr' in aux:
                    mcr_loss = aux['mcr']
                else:
                    mcr_loss = torch.tensor(0.0, device=labels.device)

                total_loss = loss_main + 0.5 * loss_video_main + config['aux_loss_weights'].get('seq',0.5)*loss_seq + config['aux_loss_weights'].get('frame',0.5)*loss_frame + mcr_loss
            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            running_loss += total_loss.item() * labels.size(0)
            probs = torch.sigmoid(outputs['fusion_logit'].detach())
            preds = (probs >= 0.5).long()
            correct += (preds == labels.long()).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=total_loss.item(), acc=correct/total if total>0 else 0)

        epoch_train_loss = running_loss / total if total>0 else 0
        epoch_train_acc = correct / total if total>0 else 0
        metrics['train_loss'].append(epoch_train_loss)
        metrics['train_acc'].append(epoch_train_acc)

        # validation
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        all_preds = []
        all_labels = []
        all_probs = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation", leave=False):
                rgb, mel, labels, _paths = batch
                rgb = rgb.to(device)
                mel = mel.to(device)
                labels = labels.to(device).view(-1).float()
                outputs = model(rgb, mel)
                loss_main = criterion(outputs['fusion_logit'].view(-1), labels)
                loss_video_main = criterion(outputs['video_logit'].view(-1), labels)
                loss_seq = criterion(outputs['video_seq_logit'].view(-1), labels)
                frame_logits = outputs['frame_logits']
                labels_frame = labels.unsqueeze(1).repeat(1, frame_logits.size(1)).to(frame_logits.device)
                loss_frame = criterion(frame_logits.view(-1), labels_frame.view(-1))
                aux = outputs.get('fusion_aux', None)
                if aux is not None and isinstance(aux, dict) and 'mcr' in aux:
                    mcr_loss = aux['mcr']
                else:
                    mcr_loss = torch.tensor(0.0, device=labels.device)
                total_loss = loss_main + 0.5 * loss_video_main + config['aux_loss_weights'].get('seq',0.5)*loss_seq + config['aux_loss_weights'].get('frame',0.5)*loss_frame + mcr_loss
                running_val_loss += total_loss.item() * labels.size(0)
                probs = torch.sigmoid(outputs['fusion_logit'])
                preds = (probs >= 0.5).long()
                correct_val += (preds == labels.long()).sum().item()
                total_val += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        epoch_val_loss = running_val_loss / total_val if total_val>0 else 0
        epoch_val_acc = correct_val / total_val if total_val>0 else 0
        try:
            val_precision = precision_score(all_labels, all_preds, zero_division=0)
        except Exception:
            val_precision = 0.0
        try:
            val_recall = recall_score(all_labels, all_preds, zero_division=0)
        except Exception:
            val_recall = 0.0
        try:
            val_auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels))>1 else 0.5
        except Exception:
            val_auc = 0.5

        cm = confusion_matrix(all_labels, all_preds) if len(all_preds)>0 else np.zeros((2,2), dtype=int)
        plot_confusion_matrix(cm, epoch, save_path=config['model_save_path'])

        print(f"Epoch {epoch+1}/{config['num_epochs']}")
        print(f"  Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"  Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}")
        print(f"  Precision:  {val_precision:.4f} | Recall:    {val_recall:.4f} | AUC: {val_auc:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        scheduler.step(epoch_val_loss)
        metrics['val_loss'].append(epoch_val_loss)
        metrics['val_acc'].append(epoch_val_acc)
        metrics['val_precision'].append(val_precision)
        metrics['val_recall'].append(val_recall)
        metrics['val_auc'].append(val_auc)

        # save best
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_wts = copy.deepcopy(model.state_dict())
            os.makedirs(config['model_save_path'], exist_ok=True)
            torch.save({'epoch': epoch, 'model_state_dict': best_wts, 'optimizer_state_dict': optimizer.state_dict(), 'config': config}, os.path.join(config['model_save_path'], 'best_multimodal.pth'))

    # load best
    model.load_state_dict(best_wts)
    plot_metrics(metrics, save_path=config['model_save_path'])
    return model, metrics

# ---------------------------
# Main runner
# ---------------------------

def main():
    # Hardcoded arguments (no CLI parsing)
    data_root = '/kaggle/input/polyglotfake-rs/PolyGlotFake'
    CONFIG['data_root'] = data_root
    CONFIG['num_epochs'] = 3
    CONFIG['batch_size'] = 8
    config = CONFIG

    if data_root is None:
        data_root = config['data_root']
    video_paths, labels = collect_videos(data_root)
    train_paths, val_paths, train_labels, val_labels = train_test_split(video_paths, labels, test_size=0.2, random_state=config['random_seed'], stratify=labels)

    normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((config['frame_size'], config['frame_size'])),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        transforms.ToTensor(),
        normalize
    ])
    val_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((config['frame_size'], config['frame_size'])),
        transforms.ToTensor(),
        normalize
    ])

    train_ds = MultiModalDataset(train_paths, train_labels, config, transform=train_transform)
    val_ds = MultiModalDataset(val_paths, val_labels, config, transform=val_transform)

    class_weights = get_class_weights(train_labels)
    sample_weights = torch.tensor([class_weights[int(l)] for l in train_labels])
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], sampler=sampler, num_workers=config['num_workers'], pin_memory=pin_memory, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'], pin_memory=pin_memory, drop_last=False)

    model = MultiModalDetector(config, audio_n_mels=config['audio_n_mels'])
    model, metrics = train_multimodal(model, train_loader, val_loader, config)
    torch.save(model.state_dict(), os.path.join(config['model_save_path'], 'final_multimodal.pth'))
    print('Training complete. Model saved.')

if __name__ == '__main__':
    main()
